In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
torch.manual_seed(1337)
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

In [3]:
text = ""
with open("../data/dataset.txt", "r", encoding="utf-8") as file:
    text = file.read()

In [4]:
chars = list(sorted(set(text)))
vocab_size = len(chars)

In [5]:
stoi = {}
itos = {}

for i, ch in enumerate(chars):
    stoi[ch] = i
    itos[i] = ch

In [6]:
def get_batch(data, batch_size, seq_len):
    high = len(data) - seq_len
    ix = torch.randint(0, high, (batch_size, ), device=device)

    idx = ix[:, None] + torch.arange(seq_len, device=device)
    x = data[idx]
    y = data[idx+1]

    return x, y

In [7]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.W_xh = nn.Linear(vocab_size, hidden_size, bias=True)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True)
        self.W_hy = nn.Linear(hidden_size, vocab_size, bias=True)

    def forward(self, x, hidden=None):
        batch_size, seq_len = x.shape

        if hidden is None:
            hidden = torch.zeros((batch_size, self.hidden_size), device=device)

        outputs = []
        for t in range(seq_len):
            x_t = x[:, t]
            # One-hot encoding
            x_t = F.one_hot(x_t, num_classes=self.vocab_size).float()

            hidden = torch.tanh(self.W_xh(x_t) + self.W_hh(hidden))
            logits = self.W_hy(hidden)

            outputs.append(logits)

        outputs = torch.stack(outputs, dim=1)
        return outputs, hidden

In [8]:
hidden_size = 256
model = CharRNN(vocab_size, hidden_size).to(device)

In [9]:
def loss_fn(logits, targets):
    batch_size, seq_len, vocab_size = logits.shape

    logits = logits.view(batch_size * seq_len, vocab_size)
    targets = targets.view(batch_size * seq_len)

    # Flatten Batch and Time
    loss = F.cross_entropy(logits, targets)
    return loss


epochs = 20000
batch_size = 128
seq_len = 64
learning_rate = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long, device=device)

# Training
model.train()
for epoch in range(epochs):
    x, y = get_batch(data, batch_size, seq_len)
    optimizer.zero_grad()

    logits, hidden = model(x)
    loss = loss_fn(logits, y)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if epoch % 2000 == 0:
        print(f"Epoch: {epoch}, Loss: {loss.item()}")

print(f"Epoch: {epochs}, Loss: {loss.item()}")

Epoch: 0, Loss: 4.184782028198242
Epoch: 2000, Loss: 1.7290552854537964
Epoch: 4000, Loss: 1.575486421585083
Epoch: 6000, Loss: 1.514931559562683
Epoch: 8000, Loss: 1.4623013734817505
Epoch: 10000, Loss: 1.4077039957046509
Epoch: 12000, Loss: 1.353877067565918
Epoch: 14000, Loss: 1.3876274824142456
Epoch: 16000, Loss: 1.3864320516586304
Epoch: 18000, Loss: 1.392530918121338
Epoch: 20000, Loss: 1.3944451808929443


In [10]:
@torch.no_grad()
def generate(start_text, max_tokens, temperature):
    model.eval()

    output = []
    output.append(start_text)

    start = torch.tensor([[stoi[ch] for ch in start_text]], dtype=torch.long, device=device)
    logits, hidden = model(start)

    for _ in range(max_tokens):
        # Last prediction
        logits = logits[:, -1, :] / temperature

        # Sample the next character
        probs = F.softmax(logits, dim=1)
        next_char_id = torch.multinomial(probs, num_samples=1)

        logits, hidden = model(next_char_id, hidden)
        output.append(itos[next_char_id.item()])

    return "".join(output)

In [11]:
print(generate(start_text="the", max_tokens=100, temperature=0.5))

the woman down, unto the earth.

KING RICHARD II:
Now I am the good for the fall of my heart may not li
